# Performance benchmarks

In [1]:
import time
from pathlib import Path

from annbatch import create_anndata_collection

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
import zarr
import zarrs

zarr.config.set({"codec_pipeline.path": "zarrs.ZarrsCodecPipeline"})

In [3]:
INPUT_PATH = Path("/mnt/dssfs02/lamindb/arc-ctc-tahoe100/2025-02-25/h5ad")
OUT_PATH_ZARR = Path("/dss/mcmlscratch/04/di93zer/tahoe100M_zarr_benchmark")
OUT_PATH_H5AD = Path("/dss/mcmlscratch/04/di93zer/tahoe100M_h5ad_benchmark")

## Compare store creation time zarr vs. h5ad

In [4]:
start_time = time.time()
create_anndata_collection(
    list(INPUT_PATH.glob("*.h5ad")), 
    OUT_PATH_H5AD,
    output_format="h5ad",
)
creation_time_h5ad = time.time() - start_time

100%|██████████| 48/48 [14:53:36<00:00, 1117.02s/it]  


In [5]:
start_time = time.time()
create_anndata_collection(
    list(INPUT_PATH.glob("*.h5ad")), 
    OUT_PATH_ZARR
)
creation_time_zarr = time.time() - start_time

  0%|          | 0/48 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/zarr/api/asynchronous.py:244: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
  2%|▏         | 1/48 [10:21<8:07:13, 622.00s/it]/usr/local/lib/python3.12/dist-packages/zarr/api/asynchronous.py:244: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
  4%|▍         | 2/48 [20:17<7:44:45, 606.20s/it]/usr/local/lib/python3.12/dist-packages/zarr/api/asynchronous.py:244: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
  6%|▋         | 3/48 [29:55<7:25:08, 593.51s/it]/usr/local/

In [6]:
import pandas as pd

In [7]:
res = pd.DataFrame({
    "format": ["zarr", "h5ad"],
    "store_creation_time": [
        creation_time_zarr / (60. * 60.),
        creation_time_h5ad / (60. * 60.)
    ],
})
res

,format,store_creation_time
0,zarr,6.724527
1,h5ad,15.242787


In [10]:
res.to_csv("store_creation_time.csv", index=False)